In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
import plotly.graph_objects as go  # Add this line
from plotly.subplots import make_subplots
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields


In [ ]:
ss = [
    # {'solution_folder': f"RTS-GMLC_v3.1.1s", 'VLGEN': 30, 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v4.1.1s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v_s1.2s", 'VLGEN': 1e3, 'model_type' : 'stochastic'}
]
days = range(152,153)
s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
solution_keys = ['storage_parameters','storage', 'dual_variables', 'reserve', 'energy_reserve']
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    # s_uc_name = 's_uc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_ed_name = 's_sed'
    s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
    if sol['model_type'] != 'stochastic':
        s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    else:
        s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    s_uc.append(s_uc_)
    s_ed.append(s_ed_)

    # gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    # gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s) 

    # gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    # gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s)

    # gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    # gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

s_uc = combine_solutions(s_uc)
s_ed = combine_solutions(s_ed)
# gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
# gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

for k,v in s_uc.items():
    if 'µ' in v.columns:
        s_uc[k]['model_type'] =  v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
for k,v in s_ed.items():
    if 'µ' in v.columns:
        s_ed[k]['model_type'] = v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)

# if 'µ' in gcdi_KPI_adequacy.columns: 
# #         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
#     gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
#     gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)


storage_ids = range(101,111)
for key in s_uc.keys():
    s_uc[key] = s_uc[key][s_uc[key].r_id.isin(storage_ids)]
for key in s_ed.keys():    
    s_ed[key] = s_ed[key][s_ed[key].r_id.isin(storage_ids)] 



In [ ]:
da_storage_reserve = s_uc['reserve'][s_uc['reserve'].resource == 'battery'][['hour','day','model_type','r_id','reserve_up_MW', 'reserve_down_MW']].copy()
da_storage_reserve.dropna(inplace = True) # we drop energy reserves with NaN values
da_storage_e_reserve = s_uc['energy_reserve'][s_uc['energy_reserve'].resource == 'battery'][['hour','hour_i','day','model_type','r_id','energy_reserve_up_MW', 'energy_reserve_down_MW']].copy()


In [ ]:
tuples = [(model_type, day, r_id, h_i, h) for day in da_storage_reserve.day.unique() for model_type in da_storage_reserve.model_type.unique() for r_id in da_storage_reserve.r_id.unique() for h in da_storage_reserve.hour.unique() for h_i in da_storage_reserve.hour.unique() if h_i <= h]
da_storage_cum_reserve = pd.DataFrame(tuples, columns=['model_type', 'day', 'r_id', 'hour_i', 'hour'])
da_storage_cum_reserve = da_storage_cum_reserve.merge(da_storage_reserve, left_on = ['model_type', 'day', 'r_id', 'hour'], right_on = ['model_type', 'day', 'r_id', 'hour'], how = 'left')
da_storage_cum_reserve.set_index(['model_type', 'day', 'r_id', 'hour_i', 'hour'], inplace=True)
da_storage_cum_reserve = da_storage_cum_reserve.groupby(['model_type', 'day', 'r_id', 'hour_i']).cumsum().reset_index()
da_storage_cum_reserve = da_storage_cum_reserve.rename(columns={
    'reserve_up_MW': 'cum_reserve_up_MW',
    'reserve_down_MW': 'cum_reserve_down_MW'
})


In [ ]:
group_storage = True
group_energy_envelopes = False

In [ ]:
ratios = merged_data = da_storage_cum_reserve.merge(
    da_storage_e_reserve[['day', 'r_id', 'hour_i', 'hour', 'energy_reserve_up_MW', 'energy_reserve_down_MW']],
    on=['day', 'r_id', 'hour_i', 'hour'],
    how='left'
)
if group_storage:
    ratios = ratios.groupby(['model_type','day', 'hour_i', 'hour']).sum().reset_index()
ratios['energy_reserve_up_ratio'] = ratios.energy_reserve_up_MW / ratios.cum_reserve_up_MW
ratios['energy_reserve_down_ratio'] = ratios.energy_reserve_down_MW / ratios.cum_reserve_down_MW

In [ ]:
day_ = 152
# Filter data for the specific day
day_data = ratios[ratios.day == day_]
field_to_plot = 'energy_reserve_up_ratio'
# Get unique values for faceting
r_ids = sorted(day_data['r_id'].unique())
model_types = sorted(day_data['model_type'].unique())

# Create subplots
fig = make_subplots(
    rows=len(model_types), 
    cols=len(r_ids),
    subplot_titles=[f'r_id: {r_id}' for r_id in r_ids] * len(model_types),
    row_titles=[f'{model_type}' for model_type in model_types],
    shared_xaxes=True,
    shared_yaxes=True
)

for i, model_type in enumerate(model_types):
    for j, r_id in enumerate(r_ids):
        # Filter data for this specific model_type and r_id
        subset = day_data[(day_data['model_type'] == model_type) & (day_data['r_id'] == r_id)]
        
        if not subset.empty:
            # Create pivot table for heatmap
            heatmap_data = subset.pivot_table(
                index='hour_i', 
                columns='hour', 
                values=field_to_plot,
                aggfunc='sum'
            )
            
            # Add heatmap
            fig.add_trace(
                go.Heatmap(
                    z=heatmap_data.values,
                    x=heatmap_data.columns,
                    y=heatmap_data.index,
                    # colorscale='RdYlBu_r',
                    coloraxis="coloraxis",
                    showscale=True if (i == 0 and j == len(r_ids)-1) else False,
                    colorbar=dict(title="Cum Reserve Up Ratio") if (i == 0 and j == len(r_ids)-1) else None,
                    # zmin  = vmin,
                    # zmax  = vmax    
                ),
                row=i+1, col=j+1
            )

fig.update_layout(
    title=f'Energy Reserve to Cumulative Reserve Up Ratio Heatmap - Day {day_}',
    height=500,
    width=500,
    coloraxis=dict(colorscale='RdYlBu_r')
)

fig.update_xaxes(title_text="Hour")
fig.update_yaxes(title_text="Hour_i")

fig.show()


In [ ]:
# Filter data for the specific day
day_data = ratios[ratios.day == day_]
field_to_plot = 'energy_reserve_down_ratio'
# Get unique values for faceting
r_ids = sorted(day_data['r_id'].unique())
model_types = sorted(day_data['model_type'].unique())

# Create subplots
fig = make_subplots(
    rows=len(model_types), 
    cols=len(r_ids),
    subplot_titles=[f'r_id: {r_id}' for r_id in r_ids] * len(model_types),
    row_titles=[f'{model_type}' for model_type in model_types],
    shared_xaxes=True,
    shared_yaxes=True
)

for i, model_type in enumerate(model_types):
    for j, r_id in enumerate(r_ids):
        # Filter data for this specific model_type and r_id
        subset = day_data[(day_data['model_type'] == model_type) & (day_data['r_id'] == r_id)]
        
        if not subset.empty:
            # Create pivot table for heatmap
            heatmap_data = subset.pivot_table(
                index='hour_i', 
                columns='hour', 
                values=field_to_plot,
                aggfunc='sum'
            )
            
            # Add heatmap
            fig.add_trace(
                go.Heatmap(
                    z=heatmap_data.values,
                    x=heatmap_data.columns,
                    y=heatmap_data.index,
                    # colorscale='RdYlBu_r',
                    coloraxis="coloraxis",
                    showscale=True if (i == 0 and j == len(r_ids)-1) else False,
                    colorbar=dict(title="Cum Reserve Up Ratio") if (i == 0 and j == len(r_ids)-1) else None,
                    # zmin  = vmin,
                    # zmax  = vmax    
                ),
                row=i+1, col=j+1
            )

fig.update_layout(
    title=f'Energy Reserve to Cumulative Reserve Down Ratio Heatmap - Day {day_}',
    height=500,
    width=500,
    coloraxis=dict(colorscale='RdYlBu_r')
)

fig.update_xaxes(title_text="Hour")
fig.update_yaxes(title_text="Hour_i")

fig.show()
